# Service Now Project Write-Up

This is a walk-through on how I used SQL to create a database and work with data inside of it.

First, I pulled all the data from the ServiceNow Rest API. The code is located in API_Call.py. This file does a couple of things. (1) It pulls data from the ServiceNow API using pagnation, as the call only brings in 1000 tickets at a time, (2) and writes these tickets into multiple JSON files, (3) and counts the amount of tickets. At the time of pulling these files, I had 22 JSON files. The framework of this call is writing using this link: https://github.com/simonbukin/ucsc-servicenow-basics/blob/master/Getting%20Ticket%20Information%20from%20ServiceNow.ipynb


#### This tutorial will explain how to use and manipulate the data inside Postgres. 

First thing to do would be to download Postgres using brew! I did everything on Bash on the terminal. Here's the steps I took to create my database.  

In [ ]:
-- Need to be in Bash!
chsh -s /bin/bash

-- Check if Postgres is installed
psql --version

-- Open psotgres shell
psql -U sgaheer -d postgres

-- Create a new database
psql -U sgaheer -h localhost -p 5432 -d postgres -c "CREATE DATABASE servicenow;"

-- Verify
psql -U sgaheer -h localhost -p 5432 -c "\l"

-- Connect to new database (step 1)
psql -U sgaheer -h localhost -p 5432 -d servicenow

psql -U sgaheer -d servicenow


-- Look at my table (step 2)
\dt
\d tickets

-- To modify the table as I made sure I could not accidently delete the table by revoking my privleges.
-- The following code only temporarily enables updating.   
SET default_transaction_read_only = off;
servicenow=# GRANT INSERT, UPDATE, DELETE ON TABLE tickets TO sgaheer;


ALTER TABLE tickets DROP COLUMN location_identification;



SyntaxError: invalid syntax (2924387700.py, line 1)

Then, after my table was created, I was able to put all my tickets into the Database! My file: postgresql.py did all of this work by (1) connecting to the server, (2) looping through all my JSON files, and (3) placing it into my database. My code addresses null handling, datetime inconsistencies, and other data quality issues. 

#### This tutorial will explain how I cleaned the data. 

Then, I had to clean the data. 

To clean the data first, I just want all the tickets hat either have "ResNet" as the assignment group or include "ResNet" in the comments. Any other type of ticket is useless. 

In [ ]:
DELETE FROM tickets
WHERE NOT (assignment_group = 'ResNet' OR comments LIKE '%ResNet%');

In [ ]:
SELECT COUNT(*) FROM tickets;


I only have 28785 tickets left. 

Now, I want to add a couple of more columns. 

(1) Location Identification. Where do our tickets come from?
(2) Year: Extract the year from the "Date Created" timestamp.
(3) Quarter_Created: Determine the quarter based on the month of the "Date Created" timestamp.
(4) Quarter_Ended: Determine the quarter based on the month of the "Date Closed" timestamp.
(5) Adding Flags via Text Classification. This is more complicated. 

In [ ]:
ALTER TABLE tickets
ADD COLUMN location_identification TEXT, -- we can do this w/o rcc and stevenson
ADD COLUMN location_specificity TEXT, -- we may not need this
ADD COLUMN year INT,
ADD COLUMN quarter_created TEXT,
ADD COLUMN quarter_ended TEXT,


In [ ]:
-- (1) Location Identification. Where do our tickets come from?
-- Rule-based text classification. 

-- ILIKE allows the search to be case insensitive

-- All exceptions are here, come back and see if there is a smarter way to do this!

UPDATE tickets
SET location_identification = 
    CASE -- order matters, so specify the null statements first
        WHEN comments ILIKE '%Stevenson%' AND comments ILIKE '%125%' THEN 'Unknown'
        WHEN comments ILIKE '%Stevenson%' AND comments ILIKE '%Stevenson Library%'THEN 'Unknown'
        WHEN comments ILIKE '%Stevenson%' THEN 'Stevenson'

        WHEN comments ILIKE '%Rachel Carson%' AND comments ILIKE '%258%' THEN 'Unknown' -- We want to ignore ALL cases in which these letters strung are together
        WHEN comments ILIKE '%College 8%' AND comments ILIKE '%258%' THEN 'Unknown'
        WHEN comments ILIKE '%RCC%' AND comments ILIKE '%258%' THEN 'Unknown'
        WHEN comments ILIKE '%Rachel Carson College%' THEN 'Rachel Carson College'
        WHEN comments ILIKE '%RCC%' THEN 'Rachel Carson College'
  
        WHEN comments ILIKE '%Cowell%' THEN 'Cowell'
        WHEN comments ILIKE '%Merrill%' THEN 'Merrill'
        WHEN comments ILIKE '%Crown%' THEN 'Crown'
        WHEN comments ILIKE '%C10%' THEN 'John R. Lewis'
        WHEN comments ILIKE '%College 10%' THEN 'John R. Lewis'
        WHEN comments ILIKE '%C9%' THEN 'College 9'
        WHEN comments ILIKE '%College 9%' THEN 'College 9'
        WHEN comments ILIKE '%Redwood Grove Apartments%' THEN 'Redwood Grove Apartments'
        WHEN comments ILIKE '%Porter%' THEN 'Porter'
        WHEN comments ILIKE '%Kresge%' THEN 'Kresge '        
        WHEN comments ILIKE '%Oakes%' THEN 'Oakes '
        WHEN comments ILIKE '%Family Student Housing%' THEN 'FSH Apartments'
        WHEN comments ILIKE '%FSH%' THEN 'FSH Apartments'
        WHEN comments ILIKE '%University Town Housing%' THEN 'University Town Housing'
        WHEN comments ILIKE '%UTC%' THEN 'University Town Housing'
        WHEN comments ILIKE '%Camper Parks%' THEN 'Camper Parks'
        WHEN comments ILIKE '%Provost House%' THEN 'Provost House'
        ELSE 'Unknown' -- if no match is found
 
    END;

UPDATE tickets
SET location_identification = NULL;

INC0384465

SELECT comments FROM tickets WHERE ticket_number = "INC0384465";

--ignore when: Stevenson College Casa 11, room 125, Rachel Carson College room 258
UPDATE tickets
SET location_specificity = 
    CASE
        WHEN comments ILIKE '%Apartment %' THEN 'Apartment'
        WHEN comments ILIKE '%Dorm %' THEN 'Dorm'
        ELSE 'Unknown' -- if no match is found
    END;


In [ ]:
-- (2) Year: Extract the year from the "Date Created" timestamp.

UPDATE tickets
SET year = EXTRACT(YEAR FROM date_created);

In [ ]:
-- (3) Quarter_Created: Determine the quarter based on the month of the "Date Created" timestamp.

ALTER TABLE tickets
ALTER COLUMN quarter_created TYPE TEXT,
ALTER COLUMN quarter_created TYPE TEXT;

UPDATE tickets
SET quarter_created = 
    CASE
        WHEN EXTRACT(MONTH FROM date_created) BETWEEN 9 AND 12 THEN 'Fall'   -- September to December
        WHEN EXTRACT(MONTH FROM date_created) BETWEEN 1 AND 3 THEN 'Winter'  -- January to March
        WHEN EXTRACT(MONTH FROM date_created) BETWEEN 4 AND 6 THEN 'Spring'  -- April to June
        WHEN EXTRACT(MONTH FROM date_created) BETWEEN 7 AND 9 THEN 'Summer'  -- July to September
    END;

In [ ]:
-- (4) Quarter_Ended: Determine the quarter based on the month of the "Date Closed" timestamp.

UPDATE tickets
SET quarter_ended = 
    CASE
        WHEN EXTRACT(MONTH FROM date_closed) BETWEEN 9 AND 12 THEN 'Fall'   -- September to December
        WHEN EXTRACT(MONTH FROM date_closed) BETWEEN 1 AND 3 THEN 'Winter'  -- January to March
        WHEN EXTRACT(MONTH FROM date_closed) BETWEEN 4 AND 6 THEN 'Spring'  -- April to June
        WHEN EXTRACT(MONTH FROM date_closed) BETWEEN 7 AND 9 THEN 'Summer'  -- July to September
    END;

In [ ]:
-- (5) Adding Flags. First, I have to make sure I create the values in the table first.

ALTER TABLE tickets ADD COLUMN housecall_flag INT;
ALTER TABLE tickets ADD COLUMN malware_flag INT;
ALTER TABLE tickets ADD COLUMN DMCA_flag INT;
ALTER TABLE tickets ADD COLUMN safeconnect_flag INT;
ALTER TABLE tickets ADD COLUMN device_in_office_flag INT;
ALTER TABLE tickets ADD COLUMN IoT_devices_flag INT;
ALTER TABLE tickets ADD COLUMN ethernet_port_flag INT;
ALTER TABLE tickets ADD COLUMN Mobile_Eye_flag INT;
ALTER TABLE tickets ADD COLUMN router_modem_flag INT;
ALTER TABLE tickets ADD COLUMN NOPS_or_CTS_flag INT;
ALTER TABLE tickets ADD COLUMN FSH_Family_flag INT;
ALTER TABLE tickets ADD COLUMN UCSC_Guest_flag INT;
ALTER TABLE tickets ADD COLUMN adapter_issues_flag INT;
ALTER TABLE tickets ADD COLUMN XOC_flag INT;
ALTER TABLE tickets ADD COLUMN WAP_flag INT;

In [6]:
# (5) Adding Flags. The way I did this was via regex. 
import psycopg2
import pandas as pd
import re

# Connect to your PostgreSQL database
conn = psycopg2.connect(
    dbname="servicenow",
    user="sgaheer",
    password="noches",
    host="localhost",
    port="5432"
)
cursor = conn.cursor()

# Pull ticket data
query = "SELECT ticket_number, comments FROM tickets"
cursor.execute(query)
data = pd.DataFrame(cursor.fetchall(), columns=["ticket_number", "comments"])

# Define regex patterns as a dictionary
patterns = {
    'housecall_flag': r"(?i)\b(housecall|visit|technician)\b",
    'malware_flag': r"(?i)\b(malware|antivirus)\b",
    'DMCA_flag': r"(?i)\b(DMCA)\b",
    'safeconnect_flag': r"(?i)\b(safeconnect)\b",
    'device_in_office_flag': r"(?i)\b(pick up)\b",
    'IoT_devices_flag': r"(?i)\b(ResWiFi Devices|Xbox|Nintendo Switch|PlayStation|Roku|Fire TV|Apple TV|Alexa|Google Home|smart bulb|robot vacuum)\b",
    'ethernet_port_flag': r"(?i)\b(ethernet|port|jack)\b",
    'Mobile_Eye_flag': r"(?i)\b(Mobile-Eye|Mobile Eye)\b",
    'router_modem_flag': r"(?i)\b(router|modem|configured)\b",
    'NOPS_or_CTS_flag': r"(?i)\b(NOPS|CTS|Core Technology Services|Senior Network Services)\b",
    'FSH_Family_flag': r"(?i)\b(FSH Family|Family Student Housing)\b",
    'UCSC_Guest_flag': r"(?i)\b(UCSC Guest|cruznet)\b",
    'adapter_issues_flag': r"(?i)\b(network adapter|purchase)\b",
    'XOC_flag': r"(?i)\b(Xfinity on Campus|HBO|Xfinity)\b",
    'WAP_flag': r"(?i)\b(WAP|Wireless Access Port)\b",
}

# Create binary columns based on patterns
def create_binary_column(df, pattern, column_name):
    df[column_name] = df['comments'].apply(lambda x: 1 if re.search(pattern, str(x), re.IGNORECASE) else 0)

# Apply each pattern
for col, pat in patterns.items():
    create_binary_column(data, pat, col)

# Push updated flags back to PostgreSQL
for _, row in data.iterrows():
    update_query = f"""
        UPDATE tickets
        SET {', '.join(f"{col} = %s" for col in patterns.keys())}
        WHERE ticket_number = %s
    """
    values = [row[col] for col in patterns.keys()] + [row['ticket_number']]
    cursor.execute(update_query, values)

conn.commit()
cursor.close()
conn.close()



In [ ]:
SELECT COUNT(*) 
FROM tickets 
WHERE comments IS NULL OR comments = '';

-- 5342

In [ ]:
# grabing data from postgres. 

import psycopg2
import pandas as pd
import openpyxl

# the following code connects to the postgres database. I deleted the actual credentials but this is the format.
#conn = psycopg2.connect(
    #dbname="",
    #user="",
    #password="",
    #host="",
    #port=""
#)
cursor = conn.cursor()

query = "SELECT ticket_number, date_created, date_closed, business_duration, year, quarter_created, quarter_ended, housecall_flag, malware_flag, DMCA_flag, safeconnect_flag, device_in_office_flag, IoT_devices_flag, ethernet_port_flag, Mobile_Eye_flag, router_modem_flag, NOPS_or_CTS_flag, FSH_Family_flag, UCSC_Guest_flag, adapter_issues_flag, XOC_flag, WAP_flag FROM tickets"
cursor.execute(query)

columns = [desc[0] for desc in cursor.description] # grab all the columns

data = pd.DataFrame(cursor.fetchall(), columns=columns)

# data['comments'] = data['comments'].apply(lambda x: ''.join([i if i.isprintable() else ' ' for i in str(x)])) this is only so we can fetch the comments value properly

# sample_df = data.sample(n=200, random_state=42) # picks random tickets for me to manually go in flag in order to train the bigger dataset. 

data.to_excel('/Users/sgaheer/Desktop/final_sn_data.xlsx', index=False)



-- Total Number of Tickets by Year/ Quarter

In [ ]:
# SELECT COUNT(YEAR)FROM tickets WHERE YEAR = 2011;

# shanked most of 2023 and 2024 oop. 

# servicenow=# SELECT COUNT(YEAR)FROM tickets WHERE YEAR = 2011;
# 1382

#servicenow=# SELECT COUNT(YEAR)FROM tickets WHERE YEAR = 2012;
#2419

# servicenow=# SELECT COUNT(YEAR)FROM tickets WHERE YEAR = 2013;
# 2364

# servicenow=# SELECT COUNT(YEAR)FROM tickets WHERE YEAR = 2014;
# 2927

# servicenow=# SELECT COUNT(YEAR)FROM tickets WHERE YEAR = 2015;
# 3000

# servicenow=# SELECT COUNT(YEAR)FROM tickets WHERE YEAR = 2016;
# 2531

# servicenow=# SELECT COUNT(YEAR)FROM tickets WHERE YEAR = 2017;
# 2723

# servicenow=# SELECT COUNT(YEAR)FROM tickets WHERE YEAR = 2018;
# 2357

# servicenow=# SELECT COUNT(YEAR)FROM tickets WHERE YEAR = 2019;
# 1619

# servicenow=# SELECT COUNT(YEAR)FROM tickets WHERE YEAR = 2020;
# 1219

#servicenow=# SELECT COUNT(YEAR)FROM tickets WHERE YEAR = 2021;
# 2404

# servicenow=# SELECT COUNT(YEAR)FROM tickets WHERE YEAR = 2022;
# 2189

#servicenow=# SELECT COUNT(YEAR)FROM tickets WHERE YEAR = 2023;
# 1610

# servicenow=# SELECT COUNT(YEAR)FROM tickets WHERE YEAR = 2024;
# 41

